# 03 Embeddings

This notebook builds quarter-specific GraphSAGE and Node2Vec embeddings and exports pooled datasets for later regression experiments in a separate notebook.


In [1]:
import sys
import os
from pathlib import Path

%matplotlib inline

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.models.embeddings import GNNConfig, Node2VecConfig
from src.models.embedding_pipeline import build_pooled_dataset

PROJECT_ROOT = Path().resolve().parents[1]
DATA_PATH = PROJECT_ROOT / 'src' / 'datasets'
OUTPUT_ROOT = PROJECT_ROOT / 'src' / 'data'

pd.set_option("display.max_columns", 200)

## Configuration

In [2]:
cfg_graphsage = GNNConfig(
    hidden_dims=(256, 64),
    dropout=0.3,
    lr=0.01,
    epochs=100,
    aggregation="mean",
    device="cpu",
)

cfg_node2vec = Node2VecConfig(
    embedding_dim=64,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    batch_size=128,
    lr=0.01,
    epochs=100,
    device="cpu",
)

TARGET_COL = "log_systemic_risk_label"
INCLUDE_RAW_FEATURES = False

OUTPUT_DATASET_GRAPHSAGE = PROJECT_ROOT / "src" / "data" / "embeddings" / "graphsage_srisk_dataset.parquet"
OUTPUT_DATASET_NODE2VEC = PROJECT_ROOT / "src" / "data" / "embeddings" / "node2vec_srisk_dataset.parquet"

OUTPUT_DATASET_GRAPHSAGE, OUTPUT_DATASET_NODE2VEC


(WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/graphsage_srisk_dataset.parquet'),
 WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/node2vec_srisk_dataset.parquet'))

# Build Embedding Dataset

## GraphSAGE

For each quarter, GraphSAGE is trained on that quarter's graph only. The resulting embeddings are merged with the regression target `systemic_risk_label` and optionally with the raw node features.


In [3]:
pooled_df_graphsage = build_pooled_dataset(
    config=cfg_graphsage,
    years=range(2016, 2024),
    quarters=(1, 2, 3, 4),
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    output_path=OUTPUT_DATASET_GRAPHSAGE,
)

pooled_df_graphsage.shape

Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


(145536, 69)

In [4]:
pooled_df_graphsage.head()

,bank_id,year,quarter,period,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,emb_10,emb_11,emb_12,emb_13,emb_14,emb_15,emb_16,emb_17,emb_18,emb_19,emb_20,emb_21,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,emb_32,emb_33,emb_34,emb_35,emb_36,emb_37,emb_38,emb_39,emb_40,emb_41,emb_42,emb_43,emb_44,emb_45,emb_46,emb_47,emb_48,emb_49,emb_50,emb_51,emb_52,emb_53,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63,log_systemic_risk_label
0,0,2016,1,2016Q1,-0.123012,0.439783,-0.219354,0.992198,-0.137594,0.003447,0.097438,-0.210708,0.503717,0.039488,-2.120986,-0.353718,-0.105815,-0.684061,-0.257440,-0.099961,0.323456,0.209740,-0.191735,-0.147188,-0.646923,-0.625470,-0.074970,0.328689,0.139070,-0.511518,0.628847,-0.310689,0.083685,-0.437333,-0.250554,0.437807,0.005442,-0.277936,0.042617,-0.451669,0.107042,0.434999,-0.288886,-0.590576,-0.713212,0.217243,0.484952,0.086484,1.937421,-0.406678,0.474985,-0.831121,-0.280067,0.664634,0.364761,1.039289,-0.396159,0.038841,1.271716,0.147168,-0.512127,0.251913,-0.325495,0.284435,-0.330994,-0.048910,0.253164,0.038337,5.375278
1,1,2016,1,2016Q1,-0.069649,0.309697,-0.153802,0.692588,-0.091858,-0.007179,0.060832,-0.159334,0.343962,0.018582,-1.457066,-0.242833,-0.079930,-0.466297,-0.174145,-0.076229,0.235171,0.120245,-0.114291,-0.099382,-0.435796,-0.433922,-0.049142,0.219898,0.115237,-0.355582,0.440010,-0.219997,0.064237,-0.303173,-0.165334,0.293554,-0.003324,-0.190609,0.005039,-0.297974,0.101058,0.286970,-0.191967,-0.422396,-0.490299,0.155070,0.330519,0.079651,1.334998,-0.277518,0.316904,-0.584735,-0.172659,0.452468,0.269141,0.717470,-0.283935,0.032458,0.873191,0.096204,-0.330472,0.158419,-0.202349,0.163871,-0.222982,-0.025602,0.154052,0.034873,3.044522
2,2,2016,1,2016Q1,-0.100309,0.372617,-0.176462,0.833987,-0.113588,-0.007183,0.082839,-0.192635,0.427129,0.019676,-1.780593,-0.287545,-0.093449,-0.571901,-0.211579,-0.091007,0.275922,0.153211,-0.142137,-0.116769,-0.534547,-0.520819,-0.059840,0.282851,0.132492,-0.429658,0.537703,-0.265554,0.074284,-0.369255,-0.202839,0.356796,-0.004791,-0.229238,0.018795,-0.363810,0.114501,0.359295,-0.240337,-0.498265,-0.597067,0.182186,0.402160,0.074032,1.645543,-0.341924,0.392825,-0.711673,-0.223281,0.560625,0.314164,0.878998,-0.337036,0.030034,1.069441,0.122468,-0.409539,0.198155,-0.252147,0.216346,-0.271284,-0.028970,0.199716,0.041385,4.564348
3,3,2016,1,2016Q1,-0.206293,0.418543,-0.044613,0.970216,-0.019930,0.030677,0.119493,-0.358507,0.329338,0.007061,-2.122553,-0.330993,-0.130462,-0.499001,-0.270315,-0.212996,0.341681,0.132011,-0.142907,-0.158493,-0.610681,-0.665450,0.018942,0.404334,0.098505,-0.459832,0.639125,-0.146673,0.220951,-0.301827,-0.296731,0.324679,-0.080608,-0.244575,-0.025762,-0.386685,0.256667,0.279006,-0.366423,-0.631710,-0.556012,0.235748,0.527038,0.026096,1.849669,-0.329243,0.357811,-0.870813,-0.293197,0.601348,0.471095,1.008549,-0.347399,-0.094188,1.119802,0.233984,-0.393356,0.179844,-0.197690,0.175463,-0.295096,0.000636,0.223833,-0.011273,3.637586
4,4,2016,1,2016Q1,-0.085394,0.329677,-0.161042,0.732415,-0.113954,-0.019216,0.070437,-0.173366,0.382740,0.001327,-1.560742,-0.253419,-0.090171,-0.505543,-0.195931,-0.093251,0.251611,0.122490,-0.114462,-0.092643,-0.476836,-0.464921,-0.050498,0.253353,0.136403,-0.379428,0.489680,-0.252126,0.075304,-0.328153,-0.175037,0.318899,-0.022989,-0.201092,0.010159,-0.309707,0.101090,0.315565,-0.203046,-0.441243,-0.528381,0.167204,0.351399,0.080480,1.462192,-0.300134,0.359809,-0.638421,-0.198757,0.509454,0.277987,0.789014,-0.298856,0.024819,0.942565,0.107138,-0.356615,0.167615,-0.221158,0.185113,-0.236115,-0.017136,0.173043,0.044421,3.367296


In [5]:
embedding_cols = [c for c in pooled_df_graphsage.columns if c.startswith("emb_")]
meta_cols = ["bank_id", "year", "quarter", "period", TARGET_COL]
meta_cols + embedding_cols[:5], len(embedding_cols)

(['bank_id',
  'year',
  'quarter',
  'period',
  'log_systemic_risk_label',
  'emb_0',
  'emb_1',
  'emb_2',
  'emb_3',
  'emb_4'],
 64)

In [6]:
pooled_df_graphsage.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)

count      mean       max
year quarter                           
2016 1         4548  0.719746  5.375278
     2         4548  0.714352  4.727388
     3         4548  0.711462  4.189655
     4         4548  0.709951  3.367296
2017 1         4548  0.710255  3.526361
     2         4548  0.708800  3.737670
     3         4548  0.708698  3.555348
     4         4548  0.709404  3.465736
2018 1         4548  0.713800  3.713572
     2         4548  0.711474  3.637586
     3         4548  0.709318  3.637586
     4         4548  0.709583  3.401197

## Node2Vec

Node2Vec is trained on quarter-specific graph structure using random walks. The resulting embeddings are merged with the same downstream target for comparison against GraphSAGE.


In [7]:
pooled_df_node2vec = build_pooled_dataset(
    config=cfg_node2vec,
    years=range(2016, 2024),
    quarters=(1, 2, 3, 4),
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    output_path=OUTPUT_DATASET_NODE2VEC,
)

pooled_df_node2vec.shape


Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


(145536, 69)

In [8]:
pooled_df_node2vec.head()


,bank_id,year,quarter,period,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,emb_10,emb_11,emb_12,emb_13,emb_14,emb_15,emb_16,emb_17,emb_18,emb_19,emb_20,emb_21,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,emb_32,emb_33,emb_34,emb_35,emb_36,emb_37,emb_38,emb_39,emb_40,emb_41,emb_42,emb_43,emb_44,emb_45,emb_46,emb_47,emb_48,emb_49,emb_50,emb_51,emb_52,emb_53,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63,log_systemic_risk_label
0,0,2016,1,2016Q1,-0.714611,0.748216,0.294934,7.142217,0.956990,-1.244624,1.629049,-6.092297,0.277388,-1.057188,-0.719355,2.465836,-5.681693,0.003402,1.172598,0.494383,-0.768867,0.392040,-0.426313,0.336188,-0.262961,-0.758530,-1.245355,0.103919,0.759731,-0.191490,-0.372269,2.172711,-0.830203,-0.106220,1.140626,0.476394,4.983182,-6.057057,-0.592634,-0.070940,-1.004322,-3.147805,0.772158,0.026241,0.369021,1.657245,-2.041039,0.594264,-0.939803,-0.917384,-0.332047,1.496757,-1.119344,-3.371355,0.705496,0.453852,2.418249,-0.104773,5.863484,-0.151259,-1.158058,2.682545,-0.241991,-0.100253,0.301289,-0.019786,-1.001732,0.776423,5.375278
1,1,2016,1,2016Q1,-0.062028,0.030374,0.622788,2.410795,1.160410,-2.709053,2.650272,-1.441119,0.454043,-0.152181,0.562977,2.309108,-2.364658,0.437978,0.730141,-0.024884,-0.029741,-0.330476,-0.004104,0.043191,0.031906,-1.841627,-0.198828,0.214385,-0.555261,-0.332459,0.009409,1.311366,-0.363502,-0.432710,1.118608,-0.662003,1.457213,-2.754993,-1.154519,1.017279,0.301592,-1.584818,-1.442633,0.628981,0.274528,-0.229456,-0.867008,-0.043392,-0.642038,-0.430382,-0.901412,0.298333,-0.591513,-3.450951,0.246427,0.303839,2.872975,0.695103,3.913507,0.022928,-0.707525,0.995459,0.387236,0.103463,-0.134021,-0.407141,-0.523668,-1.002203,3.044522
2,2,2016,1,2016Q1,-0.324059,0.397085,0.359277,4.472055,0.951348,-1.058557,0.811339,-1.237949,0.162286,-0.005793,-0.182419,0.080753,-2.648386,0.822416,0.108219,-1.122387,-0.060036,0.180255,0.590936,0.703621,0.344627,0.557110,-1.678978,-0.313456,0.483235,0.184274,-0.106574,0.878462,-0.014979,-0.167051,0.524099,0.211410,3.966546,-6.214141,-2.691616,-0.494758,0.138512,-2.187656,-1.083936,0.139599,0.972372,0.516043,-1.220516,0.137175,-0.096864,-1.320806,-1.059973,0.140201,-0.766311,-3.100886,-0.611759,0.252130,1.580315,-0.772804,4.428499,0.153794,-0.492687,0.938744,0.128330,-0.037333,-0.681137,-0.031982,-0.214416,0.744008,4.564348
3,3,2016,1,2016Q1,-0.677899,-0.204191,-1.028790,5.948523,0.802602,-0.804857,3.197194,-4.617163,-0.042498,0.277458,0.299796,2.151103,-5.056447,0.260007,0.275223,0.136526,-0.390333,0.172530,-0.683470,0.482115,0.571712,-0.509423,-0.148264,-0.424924,0.021691,-0.301082,0.048435,1.385681,-0.508777,0.191182,0.564193,-0.490736,4.052713,-3.136717,-1.069879,-0.509948,-0.139329,-3.291343,0.161141,-0.816665,-0.069794,-0.075255,-0.583890,0.016171,-0.249837,0.072706,-0.690539,0.182868,-0.375769,-0.918922,0.342304,-1.746460,1.928184,0.104353,2.218100,0.313037,-0.810092,1.703620,0.640277,-0.259595,-0.108013,0.151274,-0.603345,-0.504266,3.637586
4,4,2016,1,2016Q1,0.297267,-0.132957,0.709642,1.430269,-0.386756,-2.142158,2.044658,-3.985828,0.134256,0.268300,0.069148,0.973042,-0.665973,0.311543,0.966724,-0.288775,0.071868,-0.476399,-0.381762,0.678220,0.319111,-1.209056,0.166319,-0.304835,0.139452,0.188054,-0.470464,0.122684,-0.245773,0.305471,0.926707,-0.265615,2.002306,-2.420409,-2.166460,-0.445808,0.510116,-3.114030,-0.798383,-0.252990,0.553743,1.092359,-0.100266,0.202858,-0.607144,-0.051159,0.330158,0.381776,-0.854415,-1.674897,0.189218,0.447008,1.570322,-0.741448,1.102503,0.676426,-1.924984,0.416935,-2.124910,-1.049359,0.368044,-0.572587,0.484047,-0.934865,3.367296


In [9]:
embedding_cols = [c for c in pooled_df_node2vec.columns if c.startswith("emb_")]
meta_cols = ["bank_id", "year", "quarter", "period", TARGET_COL]
meta_cols + embedding_cols[:5], len(embedding_cols)


(['bank_id',
  'year',
  'quarter',
  'period',
  'log_systemic_risk_label',
  'emb_0',
  'emb_1',
  'emb_2',
  'emb_3',
  'emb_4'],
 64)

In [10]:
pooled_df_node2vec.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)


count      mean       max
year quarter                           
2016 1         4548  0.719746  5.375278
     2         4548  0.714352  4.727388
     3         4548  0.711462  4.189655
     4         4548  0.709951  3.367296
2017 1         4548  0.710255  3.526361
     2         4548  0.708800  3.737670
     3         4548  0.708698  3.555348
     4         4548  0.709404  3.465736
2018 1         4548  0.713800  3.713572
     2         4548  0.711474  3.637586
     3         4548  0.709318  3.637586
     4         4548  0.709583  3.401197

# Exported Datasets

The saved datasets contain:
- metadata: `bank_id`, `year`, `quarter`, `period`
- target: `log_systemic_risk_label`
- embeddings: `emb_0`, `emb_1`, ..., `emb_63`
- optional raw features from the node table (when `INCLUDE_RAW_FEATURES=True`)

Files written by this notebook:
- `src/data/embeddings/graphsage_srisk_dataset.parquet`
- `src/data/embeddings/node2vec_srisk_dataset.parquet`

## Storage notes and known limitations

**Format** — Parquet (columnar, compressed). Correct choice for wide tabular data at this scale (145k rows × 69 cols).

**`bank_id` is a positional integer (0–4547)**, not a real bank identifier (e.g. LEI or BIC). It is the row index from the quarterly node CSV files. It is consistent across quarters because the node files share the same ordering, but there is no way to look up a bank's name, country, or external reference from this ID alone.

**Embedding scales differ between models.** GraphSAGE embeddings are bounded roughly in [−2, 2] due to the GNN weight matrices. Node2Vec embeddings have much larger variance (values up to ±7 are common). Downstream ML pipelines should apply their own `StandardScaler` to the embedding columns before training; do not assume the two embedding sets are on the same scale.

**Temporal coherence.** Embeddings are produced via warm-starting: each quarter's model is initialised from the previous quarter's weights. This keeps the embedding space anchored across time so that pooling across quarters is meaningful. Without this, dimension *k* would mean something different in each quarter.

**Trained models are not persisted.** Only the pooled embedding tables are written to disk. If you need to embed new data or inspect what the model learned, you must retrain from scratch (warm-starting from the first quarter again).

## Next Step

Use the exported parquet files in a separate notebook for classical machine-learning regression or model comparison.
